In [2]:
import os
os.environ["PGPASSWORD"] = "aa8940aa"

import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 日本語フォント設定 (フォールバック付き)
try:
    plt.rcParams["font.family"] = "IPAGothic"
except Exception:
    plt.rcParams["font.family"] = ["Noto Sans CJK JP", "DejaVu Sans"]
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["figure.dpi"] = 120

In [3]:
from sqlalchemy import text

from db.connection import DatabaseConnection


def get_engine():
    """PostgreSQL接続エンジンを取得 (settings.yaml 使用)"""
    conn = DatabaseConnection()
    return conn.get_engine()


def load_races(engine, start, end):
    """指定期間のレースデータを取得（start/endはYYYYMMDD形式）"""
    sql = text("""
        SELECT * FROM raw.races
        WHERE (year || month_day)::int BETWEEN :start AND :end
        AND track_cd NOT BETWEEN 51 AND 59
        ORDER BY year, month_day, jyo_cd, kaiji, nichiji, race_num
    """)
    df = pd.read_sql(sql, engine, params={"start": start, "end": end})
    df["race_date"] = pd.to_datetime(df["year"].astype(str) + df["month_day"], format="%Y%m%d")
    return df


def load_entries(engine, race_ids):
    """race_idリストで出走馬データを取得"""
    if not race_ids:
        return pd.DataFrame()
    sql = text("""
        SELECT e.* FROM raw.entries e
        WHERE e.race_id = ANY(:race_ids)
        ORDER BY e.race_id, e.umaban
    """)
    return pd.read_sql(sql, engine, params={"race_ids": race_ids})


def load_entries_with_results(engine, start, end):
    """指定期間の出走馬データを取得（start/endはYYYYMMDD形式）

    raw.entries に raw.races を JOIN し、surface / race_date も取得。
    """
    sql = text("""
        SELECT e.*, r.year, r.month_day, r.surface, r.field_size
        FROM raw.entries e
        JOIN raw.races r ON e.race_id = r.race_id
        WHERE (r.year || r.month_day)::int BETWEEN :start AND :end
        AND r.track_cd NOT BETWEEN 51 AND 59
        AND e.finish_pos > 0
        ORDER BY e.race_id, e.umaban
    """)
    df = pd.read_sql(sql, engine, params={"start": start, "end": end})
    df["race_date"] = pd.to_datetime(df["year"].astype(str) + df["month_day"], format="%Y%m%d")
    return df


print("Setup complete. PROJECT_ROOT =", PROJECT_ROOT)

Setup complete. PROJECT_ROOT = C:\Users\hirom\develop\keiba-ai


In [4]:
def generate_mock_race_df(n_races: int = 100) -> pd.DataFrame:
    """テスト用の合成レースデータを生成 (DB不要)

    カラム名は load_entries_with_results の戻り値に合わせる。
    """
    np.random.seed(42)
    rows = []
    for i in range(n_races):
        n = np.random.randint(10, 18)
        for j in range(n):
            rows.append({
                "race_id": f"R{i:04d}",
                "race_date": pd.Timestamp("2020-01-01") + pd.Timedelta(days=i),
                "umaban": j + 1,
                "finish_pos": j + 1,
                "win_odds": max(1.1, np.random.lognormal(2.0, 0.8)),
                "ninki": j + 1,
                "surface": np.random.choice(["turf", "dirt"]),
                "distance_band": np.random.choice(["sprint", "mile", "intermediate", "long"]),
                "field_size": n,
            })
    return pd.DataFrame(rows)


print("Mock data generator ready.")

Mock data generator ready.


In [ ]:
# ETL: EveryDB2外部テーブル → プロジェクトスキーマ
from db.etl import run_full_etl

counts = run_full_etl(get_engine(), "20150101", "20261231")
print("ETL完了:", counts)

ETL テーブル:  67%|██████████████████████████████████████████████████████████████████████████████████████████████████████████                                                     | 4/6 [00:58<00:26, 13.10s/it]

In [ ]:
# データ検証: 各テーブルの行数確認
for schema_t in [("raw", "races"), ("raw", "entries"), ("raw", "payouts"),
                  ("odds_history", "odds_snapshots"), ("odds_history", "wide_odds"),
                  ("odds_history", "odds_time_series")]:
    cnt = pd.read_sql(text(f"SELECT count(*) FROM {schema_t[0]}.{schema_t[1]}"), get_engine()).iloc[0, 0]
    print(f"  {schema_t[0]}.{schema_t[1]:30s} {cnt:>10,} 件")